# Agentic AI and RAGs ?

Using only open source frameworks.
- Tiny KB with FAISS + FakeEmbeddings
- Free Wikipedia utility (no API keys)
- Rule-based planner
- Stub summarizer with optional tiny HF model (sshleifer/tiny-gpt2)
Run all cells top to bottom in Colab.

In [1]:
# 1. INSTALLATION
# Installing latest stable versions to ensure all internal components match.
!pip install -U -q transformers accelerate langchain langchain-community faiss-cpu wikipedia
# IMPORTANT: You MUST go to 'Runtime' -> 'Restart session' after this cell completes to apply updates.

## 1) Build the KB retriever

In [2]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import FakeEmbeddings

# --- STEP 1: DEFINE THE KNOWLEDGE BASE ---
# These Documents represent the 'internal' knowledge the AI should prioritize.
kb_docs = [
    Document(
        page_content="Agentic systems reason step-by-step about which tools to call instead of invoking tools blindly.",
        metadata={"source": "kb:agentic_concept"},
    ),
    Document(
        page_content="Retrievers fetch grounding passages from a knowledge base to avoid hallucinations.",
        metadata={"source": "kb:retrievers"},
    ),
    Document(
        page_content="Wikipedia is a broad-coverage fallback when the internal KB lacks specific answers.",
        metadata={"source": "kb:wikipedia_tip"},
    ),
    Document(
        page_content="Style Guide: Answers should be concise, professional, and typically 2-4 sentences long.",
        metadata={"source": "kb:style"},
    ),
]

# --- STEP 2: EMBEDDINGS AND VECTOR STORE ---
# Using FakeEmbeddings for demonstration to keep the notebook fast and free.
embeddings = FakeEmbeddings(size=128)

# FAISS is used to index our documents for high-speed similarity search.
vs = FAISS.from_documents(kb_docs, embeddings)

# Create a retriever that returns the top 2 most relevant documents.
retriever = vs.as_retriever(search_kwargs={"k": 2})
print("Vector Store initialized with", len(kb_docs), "documents.")

/tmp/ipykernel_17981/3130203942.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Vector Store initialized with 4 documents.


## 2) Open source external tool: Wikipedia search

In [3]:
from langchain_community.utilities import WikipediaAPIWrapper

# Initialize the Wikipedia wrapper with constraints on result count and length
wiki = WikipediaAPIWrapper(lang='en', top_k_results=2, doc_content_chars_max=1000)

def wiki_search(query: str, k: int = 2):
    """Searches Wikipedia and returns a list of result snippets for external grounding."""
    try:
        # Load search results including metadata
        results = wiki.load(query)
        snippets = []
        for doc in results:
            snippets.append({
                'title': doc.metadata.get('title'),
                'summary': doc.page_content
            })
        return snippets, None
    except Exception as e:
        # Return empty list and the error message if search fails
        return [], str(e)

# Quick validation of the search functionality
test_snip, _ = wiki_search('Python programming')
print(f"Retrieved {len(test_snip)} Wikipedia snippet(s).")

Retrieved 2 Wikipedia snippet(s).


## 3) Simple planner (rule-based)

In [4]:
kb_keywords = ['agentic', 'retriever', 'citation', 'ground', 'transparen', 'honest', 'style']

def plan(question: str):
    """Determines whether to query the internal KB or Wikipedia based on keywords."""
    # Normalize question for matching
    q_lower = question.lower()

    # IMPLEMENTATION: Check if any KB keyword exists in the user question
    if any(kw in q_lower for kw in kb_keywords):
        return {'action': 'kb'}
    else:
        return {'action': 'wiki'}

# Test the planner
print(f"Plan for 'How to ground?': {plan('How to ground answers?')}")
print(f"Plan for 'Python creator?': {plan('Who created Python?')}")

Plan for 'How to ground?': {'action': 'kb'}
Plan for 'Python creator?': {'action': 'wiki'}


## 4) Answer function with stub or tiny HF model

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from transformers import pipeline
import torch

# --- STEP 3: PROMPT DESIGN ---
# We define a structured prompt to guide the LLM's behavior.
prompt = ChatPromptTemplate.from_template("""Answer the question based ONLY on the provided context.
Context: {context}
Wikipedia Snippets: {wiki}
Question: {question}
Answer:""")

_GEN = None

def get_generator(model_id: str = 'sshleifer/tiny-gpt2'):
    """Initializes the text generation pipeline once (Singleton pattern)."""
    global _GEN
    if _GEN is None:
        # Using a tiny model to ensure it runs even on basic CPU runtimes.
        _GEN = pipeline('text-generation', model=model_id, device=-1)
    return _GEN

def answer_question(question: str, use_model: bool = True):
    """The RAG Pipeline: Logic for Planning -> Retrieval -> Generation."""
    # 1. Planning: Use the keyword planner from earlier
    pl = plan(question)

    # 2. Retrieval: Get data from either KB or Wikipedia
    kb_context = ""
    wiki_context = ""

    if pl['action'] == 'kb':
        docs = retriever.invoke(question)
        kb_context = " ".join([d.page_content for d in docs])
    else:
        snips, _ = wiki_search(question)
        wiki_context = " ".join([s['summary'] for s in snips])

    # 3. Generation
    if use_model:
        gen = get_generator()
        input_text = prompt.format(context=kb_context, wiki=wiki_context, question=question)
        # Constraints to keep output short and avoid long runtimes
        result = gen(input_text, max_new_tokens=50, num_return_sequences=1, truncation=True)
        return result[0]['generated_text'].split("Answer:")[-1].strip()

    return "[Model turned off] Information retrieved successfully."

## 5) Quick check on sample questions

In [6]:
tests = [
    "What is the core loop of an agentic system?",  # Targets KB
    "How should I handle source citations according to the style guide?", # Targets KB
    "Who is the current Prime Minister of the UK?", # Targets Wikipedia
    "What are the key principles for honest behavior in AI?" # Targets KB
]

for q in tests:
    # use_model set to True to execute the LLM chain
    ans = answer_question(q, use_model=True)
    print('-' * 20)
    print('Q:', q)
    print('Answer:', ans)

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.51MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/29 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_token

--------------------
Q: What is the core loop of an agentic system?
Answer: Boone perhapsGyMost Singapore Boone courtyard perhaps grandchildren skillet workshops Dreams grandchildren predatorsPros Booneozyg Wheels boils Boone Wheels rubbing� BendMost Medic653 prayingshows 236 omegaivedived Latepublic ReduxGy skilletozyg Wheels equate grandchildrenProsMini omega skillet Late lined Television Wheels
--------------------
Q: How should I handle source citations according to the style guide?
Answer: reborntinghibit ONEJD Hancock dispatchting heir vendors circumcisedimura Brew scalp Motoroladit Observ rebornRocket004 stairs Amph Rh Motorola vendorsditoho Brew trilogy Participation circumcised dispatchoho vendors antibioticpressShertinghibitohohibit Prob004 autonomy004 ONE trilogySher hauledpress


[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--------------------
Q: Who is the current Prime Minister of the UK?
Answer: Probhibit intermittent pawn vendors heir ESV credibility TA004ditpresshibit Participation Observ Amph Amph heir ESV Brew Amph004 trilogy dispatch ONE TASherhibit credibilityatisfreementhibit dispatchJD subst ONE antibiotic intermittent HancockJD pawnpresspress Probimura Hancock autonomy004 Hancock Amph
--------------------
Q: What are the key principles for honest behavior in AI?
Answer: 004ting Hancock pawn directly Prob TAtinghibitiken confirRocket Habit Hancock circumcised credibility stairsimurapress intermittent scalptingSher antibiotic Prob Daniel circumcised confirRocketdit directly vendors Brew Rh Prob vendorspress Rhoho reborn Hancock antibiotic Participation subst Jr reborn conservation Hancock trilogy reborn
